In [7]:
from pyspark.sql import SparkSession

In [8]:
from pyspark.sql.types import StringType, IntegerType

In [9]:
from pyspark.sql.functions import col,udf

In [10]:
import time

In [11]:
spark = SparkSession.builder.appName("DemoSpark").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:08:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
df=spark.range(0,10000000).withColumn("value",col("id")%1000)

In [ ]:
df.take(25)

In [13]:
print("Before partitioning", df.rdd.getNumPartitions())

Before partitioning 2


In [14]:
df_repartitioned=df.repartition(10)

26/06/13 06:08:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [15]:
print("After partition",df_repartitioned.rdd.getNumPartitions())

[Stage 0:>                                                          (0 + 2) / 2]

After partition 10


In [16]:
df_coalesced=df_repartitioned.coalesce(2)

In [17]:
print("After coalesced",df_coalesced.rdd.getNumPartitions())

[Stage 1:>                                                          (0 + 2) / 2]

After coalesced 2


In [18]:
df_repartitioned20=df.repartition(20)

In [19]:
df_repartitioned20.write.mode("overwrite").csv("output after partition data",header=True)

In [20]:
df.write.mode("overwrite").csv("output/original-partition-data",header=True)

In [21]:
df_coalesced5=df_repartitioned.coalesce(5)

In [22]:
df_coalesced5.write.mode("overwrite").csv("output/coalesce-partition-data",header=True)

In [23]:
#optimization and caching
optimized_df=df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [24]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [27]:
start_time=time.time()
count_uncached=optimized_df.count() #Action triggers DAG
end_time=time.time()
print(f"2. FIRST CACHED Action| Count: {count_uncached}| Time:{round(end_time-start_time,4)} seconds")

2. FIRST CACHED Action| Count: 2495000| Time:0.28 seconds


In [28]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [30]:
start_time=time.time()
count_first_cache=optimized_df.count() #Action triggers caching
end_time=time.time()
print(f"2. FIRST CACHED Action| Count: {count_first_cache}| Time:{round(end_time-start_time,4)} seconds")

2. FIRST CACHED Action| Count: 2495000| Time:0.1861 seconds


In [31]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [36]:
from pyspark import StorageLevel

# Clear the old caching state first
optimized_df.unpersist()

# Create a custom level: Memory=True, Deserialized=False (which means Serialized)
MEMORY_ONLY_SER = StorageLevel(useDisk=False, useMemory=True, useOffHeap=False, deserialized=False, replication=1)

# Persist using your custom configuration
optimized_df.persist(MEMORY_ONLY_SER)

DataFrame[id: bigint, value: bigint]

In [37]:
#udf
data=[("A",14),("B",19),("C",23),("D",11)]
df1=spark.createDataFrame(data,["Name","Age"])

In [38]:
df1.show()

[Stage 22:>                                                         (0 + 1) / 1]

+----+---+
|Name|Age|
+----+---+
|   A| 14|
|   B| 19|
|   C| 23|
|   D| 11|
+----+---+



In [43]:
def can_drive(age):
    if age<13:
        return ("can't have license and can't drive")
    elif 13<=age<18:
        return ("can't have license, but can drive")
    else:
        return ("have license and can drive")

In [44]:
udf_can_drive=udf(can_drive, StringType())

In [45]:
df_met1=df1.withColumn("Category",udf_can_drive(col("age")))
print("Method 1: Standard udf via df api")
df_met1.show()

Method 1: Standard udf via df api
+----+---+--------------------+
|Name|Age|            Category|
+----+---+--------------------+
|   A| 14|can't have licens...|
|   B| 19|have license and ...|
|   C| 23|have license and ...|
|   D| 11|can't have licens...|
+----+---+--------------------+

